# COMPARATIVA - Análisis de Calidad de Datos RENAMU 2022 con 3 Enfoques

Este notebook compara los tres enfoques desarrollados para analizar la calidad de datos del **Registro Nacional de Municipalidades (RENAMU 2022)** usando el mismo archivo fuente:

- `facil_RENAMU_2022_contexto(1).ipynb`
- `medio_RENAMU_2022_contexto(1).ipynb`
- `avanzado_RENAMU_2022_contexto(1).ipynb`

El dataset común utilizado es:

- `Base_RENAMU_2022_f.csv`

La comparativa considera dos niveles:

1. **Calidad de datos RENAMU 2022**: cálculo consolidado de las 8 dimensiones de calidad.
2. **Comparativa de implementación**: diferencias entre los notebooks fácil, medio y avanzado en estructura, modularidad, reutilización y complejidad.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import nbformat
import re
import time
import warnings

warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('Set2')
%matplotlib inline

plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11


In [ ]:
# Archivos de trabajo
CSV_PATH = 'Base_RENAMU_2022_f.csv'

NOTEBOOKS = {
    'Fácil': 'facil_RENAMU_2022_contexto(1).ipynb',
    'Medio': 'medio_RENAMU_2022_contexto(1).ipynb',
    'Avanzado': 'avanzado_RENAMU_2022_contexto(1).ipynb'
}

print('Archivo CSV:', CSV_PATH)
print('Notebooks a comparar:')
for nivel, ruta in NOTEBOOKS.items():
    print(f'  - {nivel}: {ruta}')


## 1. Carga del CSV RENAMU 2022

El archivo RENAMU utiliza separador `;`. Se limpian nombres de columnas y valores vacíos para que las validaciones funcionen de forma consistente.


In [ ]:
# Cargar datos RENAMU 2022
df = pd.read_csv(
    CSV_PATH,
    sep=';',
    encoding='utf-8-sig',
    low_memory=False
)

df.columns = df.columns.str.strip()
df = df.replace(r'^\s*$', np.nan, regex=True)

print(f'Dataset cargado: {len(df):,} registros')
print(f'Total de columnas: {len(df.columns):,}')
print('\nPrimeras columnas:')
print(list(df.columns[:20]))


In [ ]:
# Campos base usados en las reglas de calidad
columnas_clave = [
    'Año', 'idmunici', 'ccdd', 'ccpp', 'ccdi',
    'Ubigeo', 'Departamento', 'Provincia', 'Distrito', 'Tipomuni'
]

columnas_existentes = [col for col in columnas_clave if col in df.columns]
columnas_faltantes = [col for col in columnas_clave if col not in df.columns]

print('Columnas clave encontradas:')
print(columnas_existentes)

if columnas_faltantes:
    print('\nColumnas clave no encontradas:')
    print(columnas_faltantes)
else:
    print('\nTodas las columnas clave están presentes.')


## 2. Lectura de los tres notebooks

Se extraen métricas estructurales de cada notebook: número de celdas, líneas de código, funciones, clases e importaciones. Esto permite comparar el enfoque **procedimental**, **funcional** y **orientado a objetos**.


In [ ]:
def cargar_notebook(ruta):
    with open(ruta, 'r', encoding='utf-8') as f:
        return nbformat.read(f, as_version=4)

def analizar_estructura_notebook(nombre, ruta):
    nb = cargar_notebook(ruta)

    code_cells = [cell for cell in nb.cells if cell.cell_type == 'code']
    markdown_cells = [cell for cell in nb.cells if cell.cell_type == 'markdown']
    codigo_total = '\n'.join(cell.source for cell in code_cells)

    funciones = re.findall(r'^\s*def\s+([a-zA-Z_]\w*)\s*\(', codigo_total, flags=re.MULTILINE)
    clases = re.findall(r'^\s*class\s+([a-zA-Z_]\w*)\s*[\(:]', codigo_total, flags=re.MULTILINE)
    imports = re.findall(r'^\s*(?:import\s+\w+|from\s+\w+\s+import\s+.+)', codigo_total, flags=re.MULTILINE)

    lineas_codigo = [
        linea for linea in codigo_total.splitlines()
        if linea.strip() and not linea.strip().startswith('#')
    ]

    return {
        'Nivel': nombre,
        'Archivo': ruta,
        'Celdas totales': len(nb.cells),
        'Celdas markdown': len(markdown_cells),
        'Celdas código': len(code_cells),
        'Líneas de código': len(lineas_codigo),
        'Funciones definidas': len(funciones),
        'Clases definidas': len(clases),
        'Importaciones': len(imports),
        'Funciones': ', '.join(funciones[:10]) + ('...' if len(funciones) > 10 else ''),
        'Clases': ', '.join(clases[:10]) + ('...' if len(clases) > 10 else '')
    }

estructura_notebooks = pd.DataFrame([
    analizar_estructura_notebook(nivel, ruta)
    for nivel, ruta in NOTEBOOKS.items()
])

estructura_notebooks


## 3. Funciones comunes para evaluar calidad RENAMU

Las siguientes funciones consolidan reglas de calidad aplicables al CSV `Base_RENAMU_2022_f.csv`.

Las 8 dimensiones evaluadas son:

1. Completitud
2. Exactitud
3. Consistencia
4. Integridad
5. Razonabilidad
6. Oportunidad
7. Unicidad
8. Validez


In [ ]:
def normalizar_codigo(serie, ancho):
    '''Normaliza códigos geográficos conservando ceros a la izquierda.'''
    return (
        serie.astype(str)
        .str.strip()
        .str.replace(r'\.0$', '', regex=True)
        .str.replace('nan', '', regex=False)
        .str.zfill(ancho)
    )

def a_numero(serie):
    '''Convierte una serie a valor numérico, enviando errores a NaN.'''
    return pd.to_numeric(serie, errors='coerce')

def calcular_metricas_dimension(nombre, indices_problema, total):
    '''Calcula cantidad y porcentaje de registros con problemas por dimensión.'''
    indices_unicos = pd.Index(indices_problema).drop_duplicates()
    cantidad = len(indices_unicos)
    porcentaje = (cantidad / total) * 100 if total > 0 else 0
    registros_ok = total - cantidad
    porcentaje_ok = 100 - porcentaje

    return {
        'dimension': nombre,
        'problemas': cantidad,
        'porcentaje_problemas': porcentaje,
        'registros_ok': registros_ok,
        'porcentaje_ok': porcentaje_ok
    }

def unir_indices(*indices):
    '''Une índices de registros con problemas evitando duplicados.'''
    resultado = pd.Index([])
    for idx in indices:
        resultado = resultado.union(pd.Index(idx))
    return resultado


In [ ]:
def evaluar_calidad_renamu(df):
    total = len(df)
    metricas = []
    detalles = {}

    # 1. Completitud
    columnas_obligatorias = [col for col in columnas_clave if col in df.columns]
    filas_incompletas_clave = df[df[columnas_obligatorias].isnull().any(axis=1)] if columnas_obligatorias else df.iloc[[]]
    filas_incompletas_general = df[df.isnull().any(axis=1)]

    metricas.append(calcular_metricas_dimension('Completitud', filas_incompletas_general.index, total))
    detalles['Completitud'] = {
        'Filas con faltantes en cualquier columna': len(filas_incompletas_general),
        'Filas con faltantes en columnas clave': len(filas_incompletas_clave),
        'Celdas faltantes totales': int(df.isnull().sum().sum())
    }

    # 2. Exactitud
    problemas_exactitud = pd.Index([])

    if 'Año' in df.columns:
        anio_num = a_numero(df['Año'])
        anio_incorrecto = df[anio_num != 2022]
        problemas_exactitud = problemas_exactitud.union(anio_incorrecto.index)
    else:
        anio_incorrecto = df.iloc[[]]

    if 'Tipomuni' in df.columns:
        tipo_num = a_numero(df['Tipomuni'])
        tipo_invalido = df[~tipo_num.isin([1, 2])]
        problemas_exactitud = problemas_exactitud.union(tipo_invalido.index)
    else:
        tipo_invalido = df.iloc[[]]

    codigos_base = [col for col in ['idmunici', 'ccdd', 'ccpp', 'ccdi', 'Ubigeo'] if col in df.columns]
    codigos_no_numericos = pd.Series(False, index=df.index)
    for col in codigos_base:
        codigos_no_numericos = codigos_no_numericos | a_numero(df[col]).isna()
    problemas_exactitud = problemas_exactitud.union(df[codigos_no_numericos].index)

    metricas.append(calcular_metricas_dimension('Exactitud', problemas_exactitud, total))
    detalles['Exactitud'] = {
        'Año distinto de 2022': len(anio_incorrecto),
        'Tipomuni fuera de 1/2': len(tipo_invalido),
        'Códigos base no numéricos': int(codigos_no_numericos.sum())
    }

    # 3. Consistencia
    df_temp = df.copy()
    problemas_consistencia = pd.Index([])

    if all(col in df_temp.columns for col in ['ccdd', 'ccpp', 'ccdi', 'Ubigeo']):
        df_temp['ccdd_norm'] = normalizar_codigo(df_temp['ccdd'], 2)
        df_temp['ccpp_norm'] = normalizar_codigo(df_temp['ccpp'], 2)
        df_temp['ccdi_norm'] = normalizar_codigo(df_temp['ccdi'], 2)
        df_temp['ubigeo_esperado'] = df_temp['ccdd_norm'] + df_temp['ccpp_norm'] + df_temp['ccdi_norm']
        df_temp['ubigeo_norm'] = normalizar_codigo(df_temp['Ubigeo'], 6)
        inconsist_ubigeo = df_temp[df_temp['ubigeo_norm'] != df_temp['ubigeo_esperado']]
        problemas_consistencia = problemas_consistencia.union(inconsist_ubigeo.index)
    else:
        inconsist_ubigeo = df.iloc[[]]

    if all(col in df_temp.columns for col in ['idmunici', 'Ubigeo']):
        df_temp['idmunici_norm'] = normalizar_codigo(df_temp['idmunici'], 6)
        if 'ubigeo_norm' not in df_temp.columns:
            df_temp['ubigeo_norm'] = normalizar_codigo(df_temp['Ubigeo'], 6)
        inconsist_idmunici = df_temp[df_temp['idmunici_norm'] != df_temp['ubigeo_norm']]
        problemas_consistencia = problemas_consistencia.union(inconsist_idmunici.index)
    else:
        inconsist_idmunici = df.iloc[[]]

    metricas.append(calcular_metricas_dimension('Consistencia', problemas_consistencia, total))
    detalles['Consistencia'] = {
        'Ubigeo distinto de ccdd+ccpp+ccdi': len(inconsist_ubigeo),
        'idmunici distinto de Ubigeo': len(inconsist_idmunici)
    }

    # 4. Integridad
    problemas_integridad = pd.Index([])

    if 'Ubigeo' in df.columns:
        duplicados_ubigeo = df[df.duplicated(subset=['Ubigeo'], keep=False)]
        problemas_integridad = problemas_integridad.union(duplicados_ubigeo.index)
    else:
        duplicados_ubigeo = df.iloc[[]]

    if 'ccdd' in df.columns:
        ccdd_num = a_numero(df['ccdd'])
        departamentos_invalidos = df[ccdd_num.isna() | (ccdd_num < 1) | (ccdd_num > 25)]
        problemas_integridad = problemas_integridad.union(departamentos_invalidos.index)
    else:
        departamentos_invalidos = df.iloc[[]]

    if columnas_obligatorias:
        claves_nulas = df[df[columnas_obligatorias].isnull().any(axis=1)]
        problemas_integridad = problemas_integridad.union(claves_nulas.index)
    else:
        claves_nulas = df.iloc[[]]

    metricas.append(calcular_metricas_dimension('Integridad', problemas_integridad, total))
    detalles['Integridad'] = {
        'Ubigeos duplicados': len(duplicados_ubigeo),
        'Departamentos fuera de rango 1-25': len(departamentos_invalidos),
        'Claves geográficas nulas': len(claves_nulas)
    }

    # 5. Razonabilidad
    problemas_razonabilidad = pd.Index([])
    columnas_totales = [col for col in df.columns if col.endswith('_T')]
    inconsistencias_totales = pd.Series(False, index=df.index)

    for col_t in columnas_totales:
        base = col_t[:-2]
        col_m = base + '_M'
        col_h = base + '_H'
        if col_m in df.columns and col_h in df.columns:
            t = a_numero(df[col_t]).fillna(0)
            m = a_numero(df[col_m]).fillna(0)
            h = a_numero(df[col_h]).fillna(0)
            inconsistencias_totales = inconsistencias_totales | ((t - (m + h)).abs() > 0.01)

    problemas_razonabilidad = problemas_razonabilidad.union(df[inconsistencias_totales].index)

    columnas_conteo = [
        col for col in df.columns
        if re.search(r'(_T$|_M$|_H$|_NM$|_NH$|_CM$|_CH$|_LM$|_LH$|_VM$|_VH$)', col)
    ]
    valores_negativos = pd.Series(False, index=df.index)
    for col in columnas_conteo:
        valores_negativos = valores_negativos | (a_numero(df[col]) < 0)

    problemas_razonabilidad = problemas_razonabilidad.union(df[valores_negativos].index)

    metricas.append(calcular_metricas_dimension('Razonabilidad', problemas_razonabilidad, total))
    detalles['Razonabilidad'] = {
        'Filas con totales que no coinciden con M+H': int(inconsistencias_totales.sum()),
        'Filas con conteos negativos': int(valores_negativos.sum()),
        'Columnas tipo total evaluadas': len(columnas_totales)
    }

    # 6. Oportunidad
    # En RENAMU 2022 la oportunidad se evalúa contra el periodo esperado de levantamiento.
    if 'Año' in df.columns:
        anio_num = a_numero(df['Año'])
        problemas_oportunidad = df[(anio_num != 2022) | anio_num.isna()].index
    else:
        problemas_oportunidad = df.index

    metricas.append(calcular_metricas_dimension('Oportunidad', problemas_oportunidad, total))
    detalles['Oportunidad'] = {
        'Registros fuera del periodo 2022': len(problemas_oportunidad)
    }

    # 7. Unicidad
    duplicados_exactos = df[df.duplicated(keep=False)]
    problemas_unicidad = pd.Index(duplicados_exactos.index)

    if 'Ubigeo' in df.columns:
        duplicados_clave = df[df.duplicated(subset=['Ubigeo'], keep=False)]
        problemas_unicidad = problemas_unicidad.union(duplicados_clave.index)
    else:
        duplicados_clave = df.iloc[[]]

    metricas.append(calcular_metricas_dimension('Unicidad', problemas_unicidad, total))
    detalles['Unicidad'] = {
        'Duplicados exactos': len(duplicados_exactos),
        'Duplicados por Ubigeo': len(duplicados_clave)
    }

    # 8. Validez
    problemas_validez = pd.Index([])

    if 'Ubigeo' in df.columns:
        ubigeo_norm = normalizar_codigo(df['Ubigeo'], 6)
        ubigeo_invalido = df[~ubigeo_norm.str.match(r'^\d{6}$', na=False)]
        problemas_validez = problemas_validez.union(ubigeo_invalido.index)
    else:
        ubigeo_invalido = df.iloc[[]]

    longitudes = {
        'ccdd': 2,
        'ccpp': 2,
        'ccdi': 2,
        'idmunici': 6
    }
    codigos_longitud_invalida = pd.Series(False, index=df.index)
    for col, ancho in longitudes.items():
        if col in df.columns:
            codigos_longitud_invalida = codigos_longitud_invalida | ~normalizar_codigo(df[col], ancho).str.match(rf'^\d{{{ancho}}}$', na=False)

    problemas_validez = problemas_validez.union(df[codigos_longitud_invalida].index)

    if 'Tipomuni' in df.columns:
        tipo_num = a_numero(df['Tipomuni'])
        tipomuni_invalido = df[~tipo_num.isin([1, 2])]
        problemas_validez = problemas_validez.union(tipomuni_invalido.index)
    else:
        tipomuni_invalido = df.iloc[[]]

    metricas.append(calcular_metricas_dimension('Validez', problemas_validez, total))
    detalles['Validez'] = {
        'Ubigeo no válido': len(ubigeo_invalido),
        'Códigos con longitud inválida': int(codigos_longitud_invalida.sum()),
        'Tipomuni no válido': len(tipomuni_invalido)
    }

    return pd.DataFrame(metricas), detalles


## 4. Ejecución consolidada de calidad de datos

Se calculan las 8 dimensiones una sola vez sobre el CSV RENAMU 2022. Esta salida sirve como referencia común para comparar los tres notebooks.


In [ ]:
inicio = time.perf_counter()
df_metricas, detalles_dimensiones = evaluar_calidad_renamu(df)
tiempo_calidad = time.perf_counter() - inicio

print(f'Tiempo de evaluación consolidada: {tiempo_calidad:.4f} segundos')
df_metricas


In [ ]:
# Detalle por dimensión
for dimension, detalle in detalles_dimensiones.items():
    print(f'\n{dimension.upper()}')
    for k, v in detalle.items():
        print(f'  - {k}: {v:,}' if isinstance(v, int) else f'  - {k}: {v}')


## 5. Comparativa descriptiva de las 8 dimensiones

Esta sección muestra el resumen comparativo de problemas y registros correctos por dimensión de calidad.


In [ ]:
print('RESUMEN COMPARATIVO - DIMENSIONES DE CALIDAD RENAMU 2022')
print(df_metricas[['dimension', 'problemas', 'porcentaje_problemas', 'registros_ok', 'porcentaje_ok']].to_string(index=False))

print('\nESTADÍSTICAS DESCRIPTIVAS')
print(f"Promedio de registros con problemas: {df_metricas['problemas'].mean():,.0f}")
print(f"Mediana: {df_metricas['problemas'].median():,.0f}")
print(f"Desviación estándar: {df_metricas['problemas'].std():,.0f}")
print(f"Mínimo: {df_metricas['problemas'].min():,.0f}")
print(f"Máximo: {df_metricas['problemas'].max():,.0f}")
print(f"Porcentaje promedio de problemas: {df_metricas['porcentaje_problemas'].mean():.2f}%")
print(f"Porcentaje promedio de calidad: {df_metricas['porcentaje_ok'].mean():.2f}%")


## 6. Visualizaciones de calidad de datos

Las siguientes gráficas permiten identificar las dimensiones con más problemas y las dimensiones con mejor calidad.


In [ ]:
# Gráfico 1: Barras horizontales - porcentaje de problemas
fig, ax = plt.subplots(figsize=(14, 8))
colors = sns.color_palette('RdYlGn_r', len(df_metricas))

orden = df_metricas.sort_values('porcentaje_problemas', ascending=True)

ax.barh(orden['dimension'], orden['porcentaje_problemas'], color=colors)
ax.set_xlabel('Porcentaje de registros con problemas (%)', fontsize=12, fontweight='bold')
ax.set_title('Comparativa de problemas por dimensión de calidad - RENAMU 2022', fontsize=14, fontweight='bold', pad=20)
ax.grid(axis='x', alpha=0.3)

for i, val in enumerate(orden['porcentaje_problemas']):
    ax.text(val + 0.2, i, f'{val:.2f}%', va='center')

plt.tight_layout()
plt.show()


In [ ]:
# Gráfico 2: Barras apiladas - OK vs problemas
fig, ax = plt.subplots(figsize=(14, 8))

x = range(len(df_metricas))
ax.barh(x, df_metricas['porcentaje_ok'], label='Sin problemas', color='#2ecc71')
ax.barh(
    x,
    df_metricas['porcentaje_problemas'],
    left=df_metricas['porcentaje_ok'],
    label='Con problemas',
    color='#e74c3c'
)

ax.set_yticks(x)
ax.set_yticklabels(df_metricas['dimension'])
ax.set_xlabel('Porcentaje (%)', fontsize=12, fontweight='bold')
ax.set_title('Distribución de calidad por dimensión - RENAMU 2022', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='lower right')
ax.set_xlim(0, 100)

plt.tight_layout()
plt.show()


In [ ]:
# Gráfico 3: Heatmap de severidad
severidad_data = df_metricas.copy()
severidad_data['Severidad'] = pd.cut(
    severidad_data['porcentaje_problemas'],
    bins=[-0.01, 1, 5, 20, 100],
    labels=['Baja', 'Media', 'Alta', 'Crítica']
)

fig, ax = plt.subplots(figsize=(12, 5))
severity_matrix = severidad_data[['dimension', 'porcentaje_problemas']].set_index('dimension').T

sns.heatmap(
    severity_matrix,
    annot=True,
    fmt='.2f',
    cmap='YlOrRd',
    cbar_kws={'label': 'Porcentaje de problemas'},
    ax=ax
)

ax.set_title('Mapa de calor - Severidad por dimensión', fontsize=14, fontweight='bold', pad=20)
ax.set_ylabel('')
plt.tight_layout()
plt.show()

severidad_data[['dimension', 'porcentaje_problemas', 'Severidad']]


In [ ]:
# Gráfico 4: Radar chart de calidad
from math import pi

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))

categorias = df_metricas['dimension'].tolist()
valores = df_metricas['porcentaje_ok'].tolist()
valores += valores[:1]

angulos = [n / float(len(categorias)) * 2 * pi for n in range(len(categorias))]
angulos += angulos[:1]

ax.plot(angulos, valores, 'o-', linewidth=2, color='#3498db')
ax.fill(angulos, valores, alpha=0.25, color='#3498db')
ax.set_xticks(angulos[:-1])
ax.set_xticklabels(categorias)
ax.set_ylim(0, 100)
ax.set_title('Radar chart - Porcentaje de calidad por dimensión', fontsize=14, fontweight='bold', pad=30)
ax.grid(True)

plt.tight_layout()
plt.show()


## 7. Comparativa de los tres notebooks

Aquí se compara la implementación de los tres archivos: fácil, medio y avanzado. Esta comparación no reemplaza la evaluación de calidad anterior; muestra cómo cambia la estructura del código según el nivel.


In [ ]:
estructura_resumen = estructura_notebooks[
    [
        'Nivel',
        'Archivo',
        'Celdas totales',
        'Celdas markdown',
        'Celdas código',
        'Líneas de código',
        'Funciones definidas',
        'Clases definidas',
        'Importaciones'
    ]
]

estructura_resumen


In [ ]:
# Gráfico comparativo de estructura
estructura_plot = estructura_resumen.set_index('Nivel')[
    ['Celdas código', 'Líneas de código', 'Funciones definidas', 'Clases definidas']
]

ax = estructura_plot.plot(kind='bar', figsize=(14, 7))
ax.set_title('Comparativa estructural de notebooks: Fácil vs Medio vs Avanzado', fontsize=14, fontweight='bold', pad=20)
ax.set_ylabel('Cantidad')
ax.set_xlabel('Nivel')
ax.tick_params(axis='x', rotation=0)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Tabla interpretativa de enfoques
comparativa_enfoques = pd.DataFrame({
    'Nivel': ['Fácil', 'Medio', 'Avanzado'],
    'Enfoque': [
        'Procedimental / lineal',
        'Funcional / modular',
        'Programación orientada a objetos'
    ],
    'Ventaja principal': [
        'Más simple de leer y ejecutar paso a paso',
        'Permite reutilizar funciones y reducir repetición',
        'Escalable, extensible y más ordenado para proyectos grandes'
    ],
    'Mejor uso': [
        'Exploración inicial y aprendizaje',
        'Automatizar validaciones recurrentes',
        'Construir un sistema formal de calidad de datos'
    ],
    'Nivel de complejidad': [
        'Bajo',
        'Medio',
        'Alto'
    ]
})

comparativa_enfoques


In [ ]:
# Comparar cobertura esperada de dimensiones por notebook según términos encontrados en el código
dimensiones = [
    'Completitud',
    'Exactitud',
    'Consistencia',
    'Integridad',
    'Razonabilidad',
    'Oportunidad',
    'Unicidad',
    'Validez'
]

cobertura = []

for nivel, ruta in NOTEBOOKS.items():
    nb = cargar_notebook(ruta)
    texto = '\n'.join(cell.source for cell in nb.cells).lower()

    fila = {'Nivel': nivel}
    for dim in dimensiones:
        fila[dim] = 'Sí' if dim.lower() in texto else 'No'
    cobertura.append(fila)

df_cobertura = pd.DataFrame(cobertura)
df_cobertura


In [ ]:
# Heatmap de cobertura de dimensiones
cobertura_binaria = df_cobertura.set_index('Nivel').replace({'Sí': 1, 'No': 0})

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(
    cobertura_binaria,
    annot=df_cobertura.set_index('Nivel'),
    fmt='',
    cmap='Greens',
    cbar=False,
    linewidths=0.5,
    ax=ax
)

ax.set_title('Cobertura de dimensiones por notebook', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()


## 8. Conclusiones y recomendaciones

La comparación permite identificar tanto la calidad del dataset RENAMU 2022 como el tipo de implementación más conveniente según el objetivo.


In [ ]:
top_problemas = df_metricas.nlargest(3, 'porcentaje_problemas')
mejor_calidad = df_metricas.nsmallest(3, 'porcentaje_problemas')

print('CONCLUSIONES SOBRE EL DATASET RENAMU 2022')

print('\nDIMENSIONES CON MAYOR PROBLEMÁTICA:')
for i, (_, row) in enumerate(top_problemas.iterrows(), start=1):
    print(f"  {i}. {row['dimension']}: {row['porcentaje_problemas']:.2f}% de registros afectados")

print('\nDIMENSIONES CON MEJOR CALIDAD:')
for i, (_, row) in enumerate(mejor_calidad.iterrows(), start=1):
    print(f"  {i}. {row['dimension']}: {row['porcentaje_ok']:.2f}% de registros correctos")

print('\nCONCLUSIONES SOBRE LOS NOTEBOOKS:')
print('  1. El notebook Fácil es recomendable para explicación paso a paso y revisión manual.')
print('  2. El notebook Medio mejora la reutilización mediante funciones.')
print('  3. El notebook Avanzado es el más adecuado si se busca escalar el análisis o convertirlo en un sistema reutilizable.')
print('  4. La comparativa permite usar el mismo dataset y las mismas dimensiones para evaluar resultados de forma homogénea.')

print('\nRECOMENDACIONES:')
print('  1. Mantener una definición única de reglas de calidad para evitar diferencias entre niveles.')
print('  2. Usar el nivel Fácil para enseñanza o documentación.')
print('  3. Usar el nivel Medio para reportes recurrentes.')
print('  4. Usar el nivel Avanzado para automatización, mantenimiento y extensiones futuras.')
